## Import modules and libraries

In [ ]:
from pathlib import Path
import pycolmap
import random
import pickle

from hloc import visualization
from hloc.utils import viz_3d

## Load paths

In [ ]:
running_set = 'megaloc_superpoint_lightglue'
# running_set = 'megaloc_disk_lightglue'
# running_set = 'megaloc_aliked_lightglue'
# running_set = 'megaloc_loftr'
# running_set = 'megaloc_loma'
query_set = 'query1'

PIXLOC_ROOT = Path.cwd().parent

DB_IMG_DIR = PIXLOC_ROOT / Path("datasets/zedx_mini/images/db")
QUERY_IMG_DIR = PIXLOC_ROOT / Path("datasets/zedx_mini/images") / query_set

LOCALIZATION_DIR = PIXLOC_ROOT / Path("outputs/hloc/zedx_mini") / running_set
MODEL_DIR = LOCALIZATION_DIR / "sfm_model"
QUERY_LOC_FILE = LOCALIZATION_DIR / query_set / "query_loc.txt"

## Load the SfM model

In [ ]:
reconstruction = pycolmap.Reconstruction(str(MODEL_DIR))
print("Cameras:", len(reconstruction.cameras))
print("Images:", len(reconstruction.images))
print("Points3D:", len(reconstruction.points3D))

## Visualizing the SfM model
We visualize some of the database images with their detected keypoints.

In [ ]:
seed = random.randint(0, 1000000)

# Color the keypoints by track length: red keypoints are observed many times, blue keypoints few
visualization.visualize_sfm_2d(reconstruction, DB_IMG_DIR, n=1, seed=seed, color_by="track_length")

# Color the keypoints by visibility: blue if sucessfully triangulated, red if never matched
visualization.visualize_sfm_2d(reconstruction, DB_IMG_DIR, n=1, seed=seed, color_by="visibility")

# Color the keypoints by triangulated depth: red keypoints are far away, blue keypoints are closer
visualization.visualize_sfm_2d(reconstruction, DB_IMG_DIR, n=1, seed=seed, color_by="depth")

## Visualize the localization
We parse the localization logs and for each query image plot matches and inliers with a few database images.

In [ ]:
with open(str(QUERY_LOC_FILE) + "_logs.pkl", "rb") as f:
    logs = pickle.load(f)

In [ ]:
seed = random.randint(0, 1000000)
query_imgs = []
# query_imgs = ['1778982623808570000.png']

n = 1
if not query_imgs:
    queries = list(logs["loc"].keys())
    selected = random.Random(seed).sample(queries, min(n, len(queries)))

if reconstruction is not None:
    if not isinstance(reconstruction, pycolmap.Reconstruction):
        reconstruction = pycolmap.Reconstruction(reconstruction)

for qname in selected:
    loc = logs["loc"][qname]
    visualization.visualize_loc_from_log(
        QUERY_IMG_DIR, qname, loc, reconstruction, DB_IMG_DIR, top_k_db=1,
    )